In [33]:
import pandas as pd
from rdkit import Chem
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
import numpy as np
import random
from sklearn.model_selection import train_test_split
from itertools import product
from sklearn.model_selection import StratifiedKFold



import sys
import os
sys.path.append(os.path.abspath('../'))
from reduceGraph import mol_to_graph, graph_to_pyg_oh, reduce_graph_from_mol_oh, mol_to_pool_idx
from networks import GAT, PPGAT

In [34]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

In [35]:
#DATASET 

dataset = pd.read_csv('HIV.csv')
dataset


,smiles,activity,HIV_active
0,CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...,CI,0
1,C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...,CI,0
2,CC(=O)N1c2ccccc2Sc2c1ccc1ccccc21,CI,0
3,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,CI,0
4,O=S(=O)(O)CCS(=O)(=O)O,CI,0
...,...,...,...
41122,CCC1CCC2c3c([nH]c4ccc(C)cc34)C3C(=O)N(N(C)C)C(...,CI,0
41123,Cc1ccc2[nH]c3c(c2c1)C1CCC(C(C)(C)C)CC1C1C(=O)N...,CI,0
41124,Cc1ccc(N2C(=O)C3c4[nH]c5ccccc5c4C4CCC(C(C)(C)C...,CI,0
41125,Cc1cccc(N2C(=O)C3c4[nH]c5ccccc5c4C4CCC(C(C)(C)...,CI,0


In [36]:
dataset.HIV_active.value_counts()

HIV_active
0    39684
1     1443
Name: count, dtype: int64

In [37]:
def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_pyg_dataset(df, smiles_col, label_col):
    data_list = []
    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [38]:
HIV_dataset = dataframe_to_pyg_dataset(dataset, 'smiles', 'HIV_active')


[14:19:50] WARNING: not removing hydrogen atom without neighbors
[14:19:50] WARNING: not removing hydrogen atom without neighbors


In [39]:
#grid search
learning_rates = [1e-4, 5e-4, 1e-3]
batch_sizes = [16, 32, 64]
grid = list(product(learning_rates, batch_sizes))

In [40]:
#multi label aware stratified split


def train_eval_model(mod, dataset, in_channels, edge_attr_dim, out_channels, grid, epochs=30):
    labels = [int(data.y.item()) for data in dataset]

    train_val_indices, test_indices = train_test_split(
        list(range(len(dataset))),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    train_val_labels = [labels[i] for i in train_val_indices]

    # 10% of full = 12.5% of 80% → 0.125
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.125,
        stratify=train_val_labels,
        random_state=42
    )

    # Create subsets 
    train = [dataset[i] for i in train_indices]
    val = [dataset[i] for i in val_indices]
    test  = [dataset[i] for i in test_indices]


    #  Hyperparameter tuning 
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=batch_size)

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float().unsqueeze(1))
                loss.backward()
                optimizer.step()

        # Evaluate on val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to('cuda')
                out = model(batch)
                val_loss += criterion(out, batch.y.float().unsqueeze(1)).item() * batch.num_graphs


        val_loss /= len(val)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #  Retrain final model on train + val 
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
    final_model.load_state_dict(best_model_state)

    full_train = train + val
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to('cuda')
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float().unsqueeze(1))
            loss.backward()
            optimizer.step()

    return final_model, best_config, best_val_loss

In [41]:
in_channels = HIV_dataset[0].x.size(-1)
edge_attr_dim = HIV_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, HIV_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.001, 64)


In [42]:
def cross_validate_stratified(
    mod,
    dataset, in_channels, edge_attr_dim, out_channels,
    best_lr, best_batch_size, epochs=30, k=5
):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    import numpy as np
    import torch

    # Extract single target per sample
    y_vector = torch.stack([data.y.view(-1)[0] for data in dataset]).numpy()

    splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_aurocs = []
    all_bal_accs = []

    for fold, (train_val_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_vector)), y_vector)):
        print(f"\n Fold {fold + 1}")

        # Split data
        y_train_val = y_vector[train_val_idx]
        train_val_data = [dataset[i] for i in train_val_idx]
        test_data = [dataset[i] for i in test_idx]

        # Inner val split
        inner_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=fold)
        train_idx, val_idx = next(inner_split.split(np.zeros(len(y_train_val)), y_train_val))
        train = [train_val_data[i] for i in train_idx]
        val = [train_val_data[i] for i in val_idx]

        # Initialize model
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=best_batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=best_batch_size)
        test_loader = DataLoader(test_data, batch_size=best_batch_size)

        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)

                y_true = batch.y.float()
                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(1)

                loss = criterion(out, y_true)
                loss.backward()
                optimizer.step()

        # Evaluate on test set
        model.eval()
        y_true, y_scores = [], []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to('cuda')
                out = model(batch)

                y_scores.append(out.cpu())
                y_true.append(batch.y.cpu())

        y_scores = torch.cat(y_scores).numpy()
        y_true = torch.cat(y_true).numpy()

        # AUROC
        auroc = roc_auc_score(y_true, y_scores)
        all_aurocs.append(auroc)

        # Balanced accuracy
        y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid
        y_pred = (y_probs >= 0.5).astype(int)

        bal_acc = balanced_accuracy_score(y_true, y_pred)
        all_bal_accs.append(bal_acc)

        print(f" Fold {fold + 1} AUROC: {auroc:.4f}")
        print(f" Fold {fold + 1} Balanced Acc: {bal_acc:.4f}")

    mean_auroc = np.mean(all_aurocs)
    std_auroc = np.std(all_aurocs)

    mean_bal_acc = np.mean(all_bal_accs)
    std_bal_acc = np.std(all_bal_accs)

    print(f"\n Mean AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f" Mean Balanced Accuracy = {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")

    return (
        all_aurocs, mean_auroc, std_auroc,
        all_bal_accs, mean_bal_acc, std_bal_acc
    )

In [43]:
result = cross_validate_stratified( GAT, 
    HIV_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.7527
 Fold 1 Balanced Acc: 0.6087

 Fold 2
 Fold 2 AUROC: 0.7490
 Fold 2 Balanced Acc: 0.6076

 Fold 3
 Fold 3 AUROC: 0.7854
 Fold 3 Balanced Acc: 0.6575

 Fold 4
 Fold 4 AUROC: 0.7809
 Fold 4 Balanced Acc: 0.6320

 Fold 5
 Fold 5 AUROC: 0.7757
 Fold 5 Balanced Acc: 0.6169

 Mean AUROC = 0.7688 ± 0.0150
 Mean Balanced Accuracy = 0.6245 ± 0.0187


# RG

In [27]:

def smiles_to_rgdata(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    data = reduce_graph_from_mol_oh(mol)  # pytorch geometric RG graph from mol 
    data.y = torch.tensor([label], dtype=torch.float) #add label 
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgdata(smiles, label)
        if data is not None:
            #add smiles to dataset 
            data.smiles = smiles            
            data_list.append(data)

    return data_list

In [28]:
HIV_rg_dataset = dataframe_to_rg_pyg_dataset(dataset, 'smiles', 'HIV_active')

[22:07:56] WARNING: not removing hydrogen atom without neighbors
[22:07:56] WARNING: not removing hydrogen atom without neighbors


In [29]:

model, config, loss = train_eval_model(GAT, HIV_rg_dataset, HIV_rg_dataset[0].x.size(-1), HIV_rg_dataset[0].edge_attr.size(-1), 1, grid)

print(config )


(0.001, 64)


In [ ]:
results = cross_validate_stratified(
    GAT, 
    HIV_rg_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.7968
 Fold 1 Balanced Acc: 0.6742

 Fold 2
 Fold 2 AUROC: 0.7521
 Fold 2 Balanced Acc: 0.6262

 Fold 3
 Fold 3 AUROC: 0.7542
 Fold 3 Balanced Acc: 0.6484

 Fold 4
 Fold 4 AUROC: 0.7823
 Fold 4 Balanced Acc: 0.6244

 Fold 5
 Fold 5 AUROC: 0.7454
 Fold 5 Balanced Acc: 0.6319

 Mean AUROC = 0.7661 ± 0.0198
 Mean Balanced Accuracy = 0.6410 ± 0.0186


ValueError: too many values to unpack (expected 3)

# PPGAT

In [44]:

def smiles_to_rgnn_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    data.y = torch.tensor([label], dtype=torch.float) #add label 


    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

      # Skip if edge_attr is missing 
    if data.edge_attr is None or data.edge_attr.dim() != 2 or data.edge_attr.size(0) == 0 or data.edge_attr.size(1) == 0:
        return None
    
    if data.edge_index is None or data.edge_index.dim() != 2 or data.edge_index.size(0) == 0 or data.edge_index.size(1) == 0:
        return None


 

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col='nonstereo_aromatic_smiles', label_col='label'):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        data = smiles_to_rgnn_data(smiles, label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [46]:
HIV_ppgat_dataset = dataframe_to_rgnn_pyg_dataset(dataset, 'smiles', 'HIV_active')

[16:38:47] WARNING: not removing hydrogen atom without neighbors
[16:38:47] WARNING: not removing hydrogen atom without neighbors


In [47]:
model, config, loss = train_eval_model(PPGAT, HIV_ppgat_dataset, HIV_ppgat_dataset[0].x.size(-1), HIV_ppgat_dataset[0].edge_attr.size(-1), 1, grid)
print(config)

(0.0005, 32)


In [48]:
results = cross_validate_stratified(
    PPGAT, 
    HIV_ppgat_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8071
 Fold 1 Balanced Acc: 0.6441

 Fold 2
 Fold 2 AUROC: 0.7366
 Fold 2 Balanced Acc: 0.6405

 Fold 3
 Fold 3 AUROC: 0.7804
 Fold 3 Balanced Acc: 0.6389

 Fold 4
 Fold 4 AUROC: 0.7623
 Fold 4 Balanced Acc: 0.6219

 Fold 5
 Fold 5 AUROC: 0.7783
 Fold 5 Balanced Acc: 0.6495

 Mean AUROC = 0.7729 ± 0.0232
 Mean Balanced Accuracy = 0.6390 ± 0.0093
